In [ ]:
%cd /kaggle/working

import os
import subprocess
import sys
import torch

# ==========================================
# ⚙️ CONFIGURATION PANEL (EDIT VALUES HERE)
# ==========================================
NGROK_TOKEN = ""
STATIC_DOMAIN_API = ""
STATIC_DOMAIN_TTYD = ""
REPO_ID = "bartowski/Qwen_Qwen3.6-35B-A3B-GGUF"
MODEL_FILENAME = "Qwen_Qwen3.6-35B-A3B-Q4_K_M.gguf"
GDRIVE_FILE_ID = "1Y3LeCBy_hoK3MOszwWtaVoAXzwpgt_US"
FORCE_REBUILD = False  # Set to True to force building from source instead of unpacking archive
# ==========================================

print(f"🔥 Available GPUs in system: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"    - GPU {i}: {torch.cuda.get_device_name(i)}")

# 1. Install lightweight Python tools
print("\n⚙️ Checking basic tools (gdown, huggingface_hub, pyngrok)...")
!pip install -q gdown huggingface_hub pyngrok

server_bin = "llama.cpp/build/bin/llama-server"
archive_name = "llama.cpp.tar.gz"

# 2. Always clean up old llama.cpp folder to prevent corrupted session state from previous Kaggle runs
print("\n🧹 Cleaning up old llama.cpp directory...")
!rm -fr llama.cpp

# 3. Choose path: Unpack prebuilt archive OR clone for building from source
if not FORCE_REBUILD:
    # --- FAST PATH: Unpack prebuilt archive from Google Drive ---
    if GDRIVE_FILE_ID and GDRIVE_FILE_ID != "YOUR_FILE_ID_HERE":
        print("\n📥 Downloading prebuilt llama.cpp archive from Google Drive...")
        !gdown {GDRIVE_FILE_ID} -O {archive_name}
        
        if os.path.exists(archive_name):
            print("📦 Unpacking llama.cpp archive...")
            !apt-get update -qq && apt-get install -y -qq pigz
            !pigz -dc {archive_name} | tar -xf -
            !rm {archive_name}
            print("✅ Archive successfully unpacked!")
        else:
            print("⚠️ Failed to download archive from Google Drive, falling back to GitHub clone...")
            !git clone https://github.com/ggerganov/llama.cpp.git
    else:
        print("\n⚙️ GDRIVE_FILE_ID not set. Cloning fresh repository from GitHub...")
        !git clone https://github.com/ggerganov/llama.cpp.git
else:
    # --- REBUILD PATH: Clone fresh repo and compile from source ---
    print("\n⚠️ FORCE_REBUILD is enabled. Cloning fresh repository from GitHub for compilation...")
    !git clone https://github.com/ggerganov/llama.cpp.git

# 4. Check if binary is ready (if we unpacked an archive, it's already built; if we cloned, we need to build)
is_ready = False
if os.path.exists(server_bin) and not FORCE_REBUILD:
    os.chmod(server_bin, 0o755)
    is_ready = True

# 5. Build from source if binary is missing (e.g. fresh clone) or FORCE_REBUILD=True
if is_ready:
    print("\n✅ Prebuilt llama-server is ready and operational!")
else:
    print("\n⚙️ Starting compilation from source (T4 sm_75, C++17)...")
    !apt-get update -qq && apt-get install -y -qq ccache
    !ccache -M 2G

    # Create custom libcuda.so stub for linker
    custom_stubs_dir = "/kaggle/working/cuda_stubs"
    os.makedirs(custom_stubs_dir, exist_ok=True)
    stub_so = os.path.join(custom_stubs_dir, "libcuda.so")
    if not os.path.exists(stub_so):
        dummy_c = os.path.join(custom_stubs_dir, "dummy_cuda.c")
        with open(dummy_c, "w") as f:
            f.write("void cuInit() {} \nvoid cuCtxCreate() {} \nvoid cuDeviceGet() {}\n")
        subprocess.run(["cc", "-shared", "-fPIC", dummy_c, "-o", stub_so], check=True)

    print("🧹 Cleaning legacy CMake cache...")
    !rm -rf llama.cpp/build/CMakeCache.txt llama.cpp/build/CMakeFiles
    
    print("🔧 Applying CUDA::cuda_driver patch in ggml-cuda/CMakeLists.txt...")
    ggml_cuda_cmake = "llama.cpp/ggml/src/ggml-cuda/CMakeLists.txt"
    if os.path.exists(ggml_cuda_cmake):
        with open(ggml_cuda_cmake, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
        content = content.replace("CUDA::cuda_driver", stub_so)
        with open(ggml_cuda_cmake, "w", encoding="utf-8") as f:
            f.write(content)

    cmake_cmd = (
        f"cd llama.cpp && cmake -B build -DGGML_CUDA=ON "
        f"-DCMAKE_CUDA_ARCHITECTURES=75 "
        f"-DCMAKE_CXX_STANDARD=17 "
        f"-DCMAKE_CUDA_STANDARD=17 "
        f"-DGGML_CCACHE=ON "
        f'-DCMAKE_CUDA_FLAGS="-DCCCL_IGNORE_DEPRECATED_CPP_DIALECT" && '
        f"cmake --build build --config Release --target llama-server -j4"
    )
    result = subprocess.run(cmake_cmd, shell=True)

    if result.returncode != 0 or not os.path.exists(server_bin):
        print("\n❌ ERROR: llama-server build failed.")
        sys.exit(1)

    print("✅ llama-server successfully built!")

# ==========================================
# Download Model to /tmp
# ==========================================
from huggingface_hub import hf_hub_download

tmp_model_dir = "/tmp/models"
os.makedirs(tmp_model_dir, exist_ok=True)
model_path = os.path.join(tmp_model_dir, MODEL_FILENAME)

if os.path.exists(model_path):
    print(f"\n📂 Model already found in /tmp: {model_path}")
else:
    print(f"\n📥 Downloading model from Hugging Face ({REPO_ID}) to /tmp...")
    model_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=MODEL_FILENAME,
        local_dir=tmp_model_dir
    )
    print(f"✅ Model successfully downloaded to: {model_path}")

In [ ]:
import subprocess
import time
import threading
from pyngrok import ngrok
import os
import sys

# Fetch variables from form or fallbacks
repo_id = globals().get("REPO_ID", "bartowski/Qwen_Qwen3.6-35B-A3B-GGUF")
model_filename = globals().get("MODEL_FILENAME", "Qwen_Qwen3.6-35B-A3B-Q4_K_M.gguf")
ngrok_token = globals().get("NGROK_TOKEN", "")
static_domain_api = globals().get("STATIC_DOMAIN_API", "semichemical-costively-leonida.ngrok-free.dev")

MODEL_FILE = os.path.join("/tmp/models", model_filename)
SERVER_BIN = "llama.cpp/build/bin/llama-server"

if not os.path.exists(MODEL_FILE):
    print(f"❌ ERROR: Model file not found at {MODEL_FILE}. Run the first cell first.")
    sys.exit(1)

if not os.path.exists(SERVER_BIN):
    print(f"❌ ERROR: Server binary not found ({SERVER_BIN}). Ensure the first cell completed successfully.")
    sys.exit(1)

# 🛑 Terminate old llama-server processes and ngrok tunnels
print("🧹 Cleaning up old llama-server processes and tunnels...")
subprocess.run(["pkill", "-f", "llama-server"], check=False)
ngrok.kill()
time.sleep(1)

if ngrok_token:
    try:
        ngrok.set_auth_token(ngrok_token)
    except Exception:
        pass

# Setup dynamic linker library paths
env = os.environ.copy()
lib_paths = [
    os.path.abspath("llama.cpp/build/bin"),
    os.path.abspath("llama.cpp/build/src"),
    os.path.abspath("llama.cpp/build/common")
]
env["LD_LIBRARY_PATH"] = ":".join(lib_paths) + ":" + env.get("LD_LIBRARY_PATH", "")

# Server configuration with 128k context, q4_0 cache quantization, and session persistence
server_cmd = [
    SERVER_BIN,
    "--model", MODEL_FILE,
    "--ctx-size", "131072",                    # 128k context length
    "--cache-type-k", "q4_0",                  # Key cache quantization (4-bit)
    "--cache-type-v", "q4_0",                  # Value cache quantization (4-bit)
    "--n-gpu-layers", "99",                    # Offload all layers to GPU
    "--tensor-split", "0.5,0.5",               # Split evenly across both T4 GPUs
    "--port", "8000",
    "--host", "0.0.0.0"
]

!chmod +x {SERVER_BIN}

print("🚀 Starting llama-server (128k context, q4_0 KV cache) in background...")
server_process = subprocess.Popen(
    server_cmd, 
    env=env, 
    stdout=subprocess.PIPE, 
    stderr=subprocess.STDOUT, 
    text=True
)

# Stream logs in real-time
def stream_logs(proc):
    for line in proc.stdout:
        print(line, end="", flush=True)

threading.Thread(target=stream_logs, args=(server_process,), daemon=True).start()

print("⏳ Loading model into 2x T4 VRAM... Please wait 30 seconds...")
time.sleep(30)

# Connect ngrok tunnel for API/WebUI
try:
    public_url = ngrok.connect(8000, url=static_domain_api).public_url
except Exception:
    public_url = ngrok.connect(8000).public_url

print("\n" + "="*70)
print(f"🎉 Everything is up and running!")
print(f"💬 **Built-in WebUI for browser chats:**")
print(f"    👉 Click or open in browser: {public_url}")
print(f"\n🔌 **API for IDEs (Cursor, VS Code / Continue, Aider):**")
print(f"    👉 Base URL: {public_url}/v1")
print(f"    👉 API Key: Any string (e.g., `not-needed`)")
print(f"    👉 Model Name: `qwen`")
print("="*70 + "\n")

In [ ]:
import os
import subprocess
from pyngrok import ngrok

ngrok_token = globals().get("NGROK_TOKEN", "")
static_domain_ttyd = globals().get("STATIC_DOMAIN_TTYD", "")

# 1. Check and install ttyd if missing
ttyd_path = "/kaggle/working/ttyd"
!chmod +x {ttyd_path}
if not os.path.exists(ttyd_path):
    print("📥 Downloading ttyd...")
    !wget -nc https://github.com/tsl0922/ttyd/releases/download/1.7.3/ttyd.i686 -O {ttyd_path}
    !chmod +x {ttyd_path}

# 2. Terminate old ttyd processes and ngrok sessions
subprocess.run(["pkill", "-f", "ttyd"], check=False)

if ngrok_token:
    try:
        ngrok.set_auth_token(ngrok_token)
    except Exception:
        pass

# 3. Launch ttyd on port 9000
print("🚀 Launching ttyd terminal on port 9000...")
ttyd_cmd = [ttyd_path, "-p", "9000", "bash"]
subprocess.Popen(ttyd_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 4. Connect ngrok tunnel for ttyd
try:
    if static_domain_ttyd:
        ttyd_url = ngrok.connect(9000, url=static_domain_ttyd).public_url
    else:
        ttyd_url = ngrok.connect(9000).public_url
except Exception:
    ttyd_url = ngrok.connect(9000).public_url

print("\n" + "="*70)
print(f"💻 **Terminal (ttyd) successfully launched!**")
print(f"    👉 Browser URL: {ttyd_url}")
print(f"    *(Runs concurrently with your WebUI / API on port 8000)*")
print("="*70 + "\n")